<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/05_Resorte_R_residuos_y_rango_de_validez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 05 — El resorte: el coeficiente R, los residuos y el rango de validez de un modelo

**Laboratorio 1 · Clase 5**

**Objetivos.**

1. Decidir con criterio si el modelo lleva ordenada al origen, y contrastar los dos ajustes (O5.2).
2. Explicar qué mide $R^2$ y qué no, y **mostrar con datos propios que un $R^2$ alto no valida un
   modelo** (O5.3).
3. Traducir la estructura de los residuos en una hipótesis física verificable (O5.4).
4. Determinar y reportar el **rango de validez** del modelo (O5.5).
5. Reconocer, vía Anscombe, que estadísticos idénticos son compatibles con conjuntos de datos
   radicalmente distintos (O5.6).

**Requisitos previos:** Colabs 01 a 04. La maquinaria de ajuste (`cuadrados_minimos`,
`grafico_con_residuos`) viene del Colab 04 y acá se usa, no se construye.

> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from math import floor, log10

rng = np.random.default_rng(20260909)

def reportar(x, dx, unidad=""):
    orden  = floor(log10(abs(dx)))
    cifras = 2 if int(dx / 10**orden) in (1, 2) else 1
    dec    = max(-(orden - (cifras - 1)), 0)
    return f"({x:.{dec}f} ± {dx:.{dec}f}) {unidad}".strip()

def cuadrados_minimos(x, y):
    '''Del Colab 04. Ajuste lineal no ponderado.'''
    x, y = np.asarray(x, float), np.asarray(y, float)
    N = len(x)
    Sx, Sy = x.sum(), y.sum(); Sxx, Sxy = (x*x).sum(), (x*y).sum()
    Delta = N*Sxx - Sx**2
    a = (N*Sxy - Sx*Sy) / Delta
    b = (Sxx*Sy - Sx*Sxy) / Delta
    s_y = np.sqrt(((y - (a*x + b))**2).sum() / (N - 2))
    return a, b, s_y*np.sqrt(N/Delta), s_y*np.sqrt(Sxx/Delta), s_y

def recta(x, a, b):        return a*x + b
def por_el_origen(x, a):   return a*x

def grafico_con_residuos(x, y, modelo, popt, yerr=None, xlabel='x', ylabel='y',
                         titulo='', etiqueta_modelo='ajuste'):
    '''Del Colab 04. Figura estándar: datos + modelo arriba, residuos abajo.'''
    x, y = np.asarray(x, float), np.asarray(y, float)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.6), sharex=True,
                                   gridspec_kw={'height_ratios': [3, 1]})
    xx = np.linspace(x.min(), x.max(), 400)
    ax1.errorbar(x, y, yerr=yerr, fmt='o', ms=5, capsize=3, label='datos')
    ax1.plot(xx, modelo(xx, *popt), 'crimson', lw=1.8, label=etiqueta_modelo)
    ax1.set_ylabel(ylabel); ax1.set_title(titulo, fontsize=11)
    ax1.grid(alpha=0.3); ax1.legend()
    r = y - modelo(x, *popt)
    ax2.axhline(0, color='k', lw=1)
    ax2.errorbar(x, r, yerr=yerr, fmt='o', ms=5, capsize=3)
    ax2.set_xlabel(xlabel); ax2.set_ylabel('residuos')
    ax2.grid(alpha=0.3); fig.tight_layout()
    return fig, (ax1, ax2)

---
## 1. La medición, y una advertencia sobre el rango

Elongación del resorte en función de la masa colgada. Dos indicaciones que no son burocráticas:

**Arrancá desde masas chicas.** Ahí está el fenómeno que buscamos hoy.

**No salgas del rango elástico.** Es tentador estirar hasta que la curva se doble: se ve espectacular
y se entiende de una. El problema es que la deformación plástica es **acumulativa e invisible**. El
resorte sigue funcionando, pero con otro $k$ y con una elongación residual, y vos no tenés forma de
saber en qué momento pasó. Las Clases 5, 6, 9 y 10 encadenan el **mismo** resorte: si lo pasás de
rosca hoy, la comparación entre el $k$ estático y el $k$ dinámico de la Clase 9 deja de tener
sentido, y no vas a saber por qué.

**Identificá tu resorte y conservalo.**

In [ ]:
# ============ DATOS DE EJEMPLO — REEMPLAZAR POR LOS PROPIOS ============
g = 9.81
k_verdadero  = 12.0        # N/m
F0_verdadero = 0.15        # N — tensión inicial de bobinado (ver Sección 2)

masa   = np.array([0.050, 0.100, 0.150, 0.200, 0.250, 0.300, 0.350, 0.400])   # kg
sigma_x = 0.10                                                                # cm

x_true = (masa*g - F0_verdadero) / k_verdadero * 100        # cm
elong  = x_true + rng.normal(0, sigma_x, len(masa))
np.savetxt('resorte.txt', np.c_[masa, elong], fmt='%.4f')
# =======================================================================

masa, elong = np.loadtxt('resorte.txt', unpack=True)
err = np.full(len(masa), sigma_x)

for m_, x_ in zip(masa, elong):
    print(f"m = {m_:.3f} kg   Δx = {reportar(x_, sigma_x, 'cm')}")

---
## 2. ¿Con ordenada al origen o sin ella?

La ley de Hooke se escribe $F = k\,\Delta x$. Con $F = mg$, eso predice

$$ \Delta x = \frac{g}{k}\,m $$

es decir, una recta **que pasa por el origen**: sin masa, sin elongación. Suena obvio.

Ajustemos las dos versiones sobre los mismos datos y comparemos.

In [ ]:
# (a) forzado por el origen
a0, _ = curve_fit(por_el_origen, masa, elong)
k0 = g / (a0[0]/100)

# (b) con ordenada al origen libre
a1, b1, sa1, sb1, sy1 = cuadrados_minimos(masa, elong)
k1  = g / (a1/100)
sk1 = k1 * (sa1/a1)

print(f"(a) por el origen  : pendiente = {a0[0]:.2f} cm/kg   ->  k = {k0:.2f} N/m")
print(f"(b) con ordenada   : pendiente = {a1:.2f} ± {sa1:.2f} cm/kg  ->  "
      f"k = {reportar(k1, sk1, 'N/m')}")
print(f"                     ordenada  = {reportar(b1, sb1, 'cm')}")
print()
print(f"diferencia entre los dos k: {abs(k0-k1):.2f} N/m = "
      f"{abs(k0-k1)/sk1:.0f} veces la incerteza de k")

La ordenada al origen no es compatible con cero, y ni de lejos. Y el $k$ que sale de cada ajuste
difiere en decenas de veces su propia incerteza: **la elección del modelo cambió el resultado mucho
más que la calidad de los datos.**

Antes de explicar por qué, miremos qué habría dicho el estadístico que todo el mundo usa.

---
## 3. El coeficiente $R$: qué mide y qué no

El coeficiente de correlación lineal de Pearson,

$$ R = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum (x_i-\bar{x})^2 \sum (y_i-\bar{y})^2}} $$

mide **cuán bien los datos se alinean sobre una recta**. Eso es todo lo que mide.

In [ ]:
R = np.corrcoef(masa, elong)[0, 1]
print(f"R  = {R:.6f}")
print(f"R² = {R**2:.6f}")

$R^2 = 0{,}9999$ y pico. Con ese número, en cualquier informe del país se escribe "el resorte cumple
la ley de Hooke" y nadie pregunta nada.

Pero fijate lo que acaba de pasar: **$R$ es el mismo para los dos ajustes de la Sección 2**, porque
no depende del modelo — se calcula sólo a partir de $x$ e $y$. Uno de los dos modelos está mal y $R$
no tiene forma de saberlo.

$R$ **no** mide:

- si el modelo que elegiste es el correcto;
- causalidad;
- si el ajuste es compatible con tus barras de error (para eso está el $\chi^2$ reducido, Colab 06).

Y su cuadrado, interpretado como "fracción de varianza explicada", **sólo vale para el ajuste lineal
por cuadrados mínimos ordinarios con ordenada libre**. Fuera de ese caso pierde sentido, y en modelos
no lineales puede dar cualquier cosa — incluso negativo. Volvemos sobre eso en el Colab 10.

---
## 4. Los residuos, que sí se enteran

El residuo del punto $i$ es $r_i = y_i - f(x_i)$. Si el modelo es adecuado, deben verse como ruido
alrededor de cero. Cualquier patrón visible es el modelo diciéndote que le falta algo.

In [ ]:
fig, _ = grafico_con_residuos(masa, elong, por_el_origen, a0, yerr=err,
                              xlabel='Masa colgada $m$ [kg]', ylabel='Elongación $\\Delta x$ [cm]',
                              titulo=f'Modelo $\\Delta x = a\\,m$ (forzado por el origen)   —   $R^2$ = {R**2:.5f}',
                              etiqueta_modelo='ajuste por el origen')
plt.show()

**Mirá el panel de abajo.** Los residuos no son ruido: arrancan claramente negativos, cruzan el cero
y terminan positivos. Es una tendencia monótona, visible a simple vista, con puntos que se apartan
varias veces su barra de error.

$R^2 = 0{,}9999$ no lo vio y **no podía verlo**: $R^2$ compara el ajuste contra el modelo trivial
"todo vale el promedio", y contra ese rival cualquier recta con pendiente gana por goleada. No es un
test de bondad del modelo y nunca lo fue.

Ahora el ajuste con ordenada libre:

In [ ]:
fig, _ = grafico_con_residuos(masa, elong, recta, (a1, b1), yerr=err,
                              xlabel='Masa colgada $m$ [kg]', ylabel='Elongación $\\Delta x$ [cm]',
                              titulo='Modelo $\\Delta x = a\\,m + b$ (ordenada libre)',
                              etiqueta_modelo='ajuste con ordenada')
plt.show()

print(f"dispersión de los residuos: {sy1:.3f} cm")
print(f"tu incerteza por punto    : {sigma_x:.3f} cm")
print(f"cociente                  : {sy1/sigma_x:.2f}   (debería ser ≈ 1 si el modelo está bien)")

Ahora los residuos sí parecen ruido, y su dispersión es **del orden** de la incerteza que estimaste
por punto. Ese cociente es, en germen, el $\chi^2$ reducido del Colab 06 — de hecho es su raíz
cuadrada.

Y acordate de la Clase 3: con $N-2 = 6$ grados de libertad, $s_y$ tiene ella misma una incerteza
relativa de $1/\sqrt{2\cdot 6} \approx 29\,\%$. Así que un cociente de 1,3 es perfectamente
compatible con 1, y uno de 2,5 ya no lo sería. **Para decir eso con precisión hace falta saber
cuántos grados de libertad tenés**, y ésa es exactamente la conversación de la Clase 6.

### De la estructura de los residuos a una hipótesis física

El paso que convierte el gráfico en física es preguntarse **qué significa** esa ordenada al origen
negativa. La respuesta es concreta y verificable:

> Los resortes helicoidales de tracción se bobinan con **tensión inicial**. Las espiras vienen
> apretadas unas contra otras de fábrica, y hace falta superar una fuerza $F_0$ antes de que el
> resorte empiece a elongar. Por debajo de $F_0$, $\Delta x = 0$; por encima,
> $$ F = F_0 + k\,\Delta x $$

Reordenando: $\Delta x = (g/k)\,m - F_0/k$. La ordenada al origen **es** $-F_0/k$, y es negativa.
Eso es física del fabricado del resorte, no un artefacto de la medición ni un error del experimento.

In [ ]:
F0 = -b1/100 * k1                          # b1 está en cm
sF0 = F0 * np.sqrt((sb1/b1)**2 + (sk1/k1)**2)

print("Tensión inicial de bobinado:", reportar(F0, sF0, 'N'))
print(f"   = la fuerza de una masa de {F0/g*1000:.0f} g")
print(f"   está a {F0/sF0:.0f} σ de cero, así que no es una fluctuación")

> **Ejercicio 5.1.** Colgá masas cada vez más chicas (5 g, 10 g, 20 g) y buscá el umbral: la masa a
> partir de la cual el resorte **empieza** a elongar. ¿Coincide con la $F_0$ que acabás de obtener
> extrapolando? Éste es un test de predicción, que vale mucho más que un ajuste.
>
> **Ejercicio 5.2.** ¿Qué pasa si tu resorte es de compresión, o si es de tracción pero sin
> pretensión de fábrica? Predecí qué darían los residuos del ajuste por el origen en ese caso.

---
## 5. ¿Hasta dónde vale el modelo?

Un parámetro sin su rango de validez es un número incompleto.

El modelo $\Delta x = (g/k)m$ —sin ordenada— **no es falso**: es una buena aproximación cuando
$mg \gg F_0$. La pregunta es *cuánto* de "mucho mayor" hace falta, y si ese régimen es accesible con
este resorte.

El sesgo relativo que introduce ignorar $F_0$ es del orden de $F_0/(mg)$. Para que sea despreciable
frente a la precisión con que determinás $k$, hace falta

$$ \frac{F_0}{m g} < \frac{\sigma_k}{k} $$

In [ ]:
# ¿Qué masa haría falta para que ignorar F0 fuera inocuo?
precision_relativa = sk1 / k1
m_necesaria = F0 / (g * precision_relativa)
print(f"precisión relativa de tu k : {100*precision_relativa:.2f} %")
print(f"F0 / (m g) para tu masa máxima ({masa.max()*1000:.0f} g): "
      f"{100*F0/(g*masa.max()):.1f} %")
print(f"-> haría falta m > {m_necesaria:.1f} kg  para que el sesgo quedara por debajo "
      f"de tu propia precisión")
print(f"   (elongación que eso implicaría: {m_necesaria*g/k_verdadero*100:.0f} cm)")
print()

print(f"{'desde':>7s} {'N':>3s} {'k (por el origen)':>19s} {'k (ordenada libre)':>26s} "
      f"{'ordenada':>18s}")
print("-" * 78)
for i in range(0, len(masa) - 2):
    m_, x_ = masa[i:], elong[i:]
    aa, _ = curve_fit(por_el_origen, m_, x_)
    kk0 = g / (aa[0]/100)
    a_, b_, sa_, sb_, _ = cuadrados_minimos(m_, x_)
    kk1 = g / (a_/100); skk1 = kk1*(sa_/a_)
    print(f"{m_[0]*1000:5.0f} g {len(m_):3d} {kk0:17.2f}   "
          f"{reportar(kk1, skk1, 'N/m'):>24s} {reportar(b_, sb_, 'cm'):>18s}")

**El primer resultado es el importante y probablemente no es el que esperabas:** la masa que haría
falta para poder ignorar $F_0$ está muy por encima de lo que este resorte tolera. Con esta
combinación de $k$, $F_0$ y precisión alcanzada, **el modelo simple nunca llega a ser adecuado en el
rango accesible**. No es que valga "de tal masa en adelante": no vale, y punto.

Ahora leé la tabla, que confirma lo mismo por otro camino. A medida que se sacan los puntos de masa
chica, el $k$ del ajuste forzado por el origen se mueve **muy poco** y nunca llega al valor que da el
modelo con ordenada. Es decir: recortar el conjunto no te salva del sesgo, sólo te deja con menos
datos.

Fijate en cambio que el $k$ del ajuste con ordenada libre **no se mueve** dentro de su incerteza. Ésa
es la firma del modelo correcto: **la estabilidad del parámetro frente al recorte del conjunto es un
diagnóstico de modelo, es gratis, y no requiere conocer el valor verdadero de nada.**

> Lo que se informa en un resultado no es sólo $k \pm \sigma_k$: es $k \pm \sigma_k$ **en tal rango de
> cargas, con tal modelo**. Sin eso, el número no es reproducible.

> **Ejercicio 5.3.** Rehacé el cálculo de la masa necesaria suponiendo que tu $k$ tuviera una
> precisión del 5 % en lugar de la que tiene. ¿Se vuelve accesible el régimen? Sacá la conclusión
> incómoda: **cuanto mejor medís, más modelos aproximados dejan de servirte.** Un experimento más
> preciso no sólo da un número mejor: exige un modelo mejor.

---
## 6. El cuarteto de Anscombe

Cuatro conjuntos publicados por F. J. Anscombe (*The American Statistician* **27**(1), 17–21, 1973).
Tienen —hasta la segunda o tercera cifra— el mismo promedio en $x$ y en $y$, la misma varianza, la
misma recta de ajuste y el mismo $R$.

In [ ]:
x123 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], float)
y1 = np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])
y2 = np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])
y3 = np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])
x4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], float)
y4 = np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])
conjuntos = [(x123, y1, 'I'), (x123, y2, 'II'), (x123, y3, 'III'), (x4, y4, 'IV')]

print(f"{'conj.':<7}{'x̄':>7}{'ȳ':>8}{'pendiente':>12}{'ordenada':>11}{'R':>9}{'R²':>8}")
print("-"*54)
for xa, ya, nom in conjuntos:
    pa, _ = curve_fit(recta, xa, ya)
    r = np.corrcoef(xa, ya)[0, 1]
    print(f"{nom:<7}{xa.mean():7.2f}{ya.mean():8.2f}{pa[0]:12.3f}{pa[1]:11.3f}{r:9.3f}{r**2:8.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
xx = np.linspace(3, 20, 100)
for ax, (xa, ya, nom) in zip(axes.ravel(), conjuntos):
    pa, _ = curve_fit(recta, xa, ya)
    ax.plot(xa, ya, 'o', ms=6)
    ax.plot(xx, recta(xx, *pa), 'crimson', lw=1.5)
    ax.set_title(f'Conjunto {nom}   (R = {np.corrcoef(xa, ya)[0,1]:.3f})', fontsize=10)
    ax.set_xlim(2, 20); ax.set_ylim(2, 14); ax.grid(alpha=0.3)
fig.suptitle('Cuarteto de Anscombe: mismos estadísticos, cuatro situaciones distintas', y=1.0)
fig.tight_layout(); plt.show()

Leamos los cuatro:

- **I** — lo que uno espera: relación lineal con ruido. El ajuste es apropiado.
- **II** — la relación es claramente curva. El modelo lineal es incorrecto, y $R$ no se entera.
- **III** — relación lineal perfecta arrastrada por **un solo punto atípico**. La pendiente reportada
  no describe a los otros diez datos.
- **IV** — todos los $x$ valen 8 salvo uno. La "pendiente" está determinada por un único punto: si
  ese dato tuviera un error de tipeo, cambiaría todo el resultado.

En los cuatro casos, informar "$R = 0{,}82$, el ajuste es bueno" sería igual de defendible — y en
tres de los cuatro, igual de falso.

> **Ejercicio 5.10.** Graficá los residuos de los cuatro conjuntos con `grafico_con_residuos`. ¿En
> cuáles el panel de residuos te habría avisado, y en cuáles no? El IV es el interesante.

### Correlación cero no significa independencia

$R$ mide relación **lineal**. Puede haber una dependencia funcional exacta y determinista con
$R \approx 0$.

In [ ]:
xs = np.linspace(-3, 3, 200)
ys = xs**2                      # dependencia perfecta, sin ruido
print(f"R entre x y x² (simétrico alrededor de 0) = {np.corrcoef(xs, ys)[0,1]:.2e}")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(xs, ys, '.', ms=4)
ax.set_xlabel('x'); ax.set_ylabel('y = x²')
ax.set_title('R ≈ 0 y sin embargo y está completamente determinada por x')
ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

---
## 7. *(Optativo)* La banda elástica: cuando el modelo ni siquiera es una función

Repetí la serie con una banda de goma, **subiendo** la carga hasta el máximo y después **bajándola**
pasando por los mismos valores. Anotá las dos ramas por separado.

In [ ]:
# ============ DATOS DE EJEMPLO — REEMPLAZAR POR LOS PROPIOS ============
m_banda = np.array([0.050, 0.100, 0.150, 0.200, 0.250, 0.300])
subida  = 2.0*m_banda*100 + 8.0*(m_banda*100)**2/100 + rng.normal(0, 0.3, len(m_banda))
bajada  = subida + np.array([2.6, 2.4, 1.9, 1.3, 0.6, 0.0]) + rng.normal(0, 0.3, len(m_banda))
# =======================================================================

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(m_banda, subida, 'o-', ms=6, label='carga (subiendo)')
ax.plot(m_banda, bajada, 's--', ms=6, label='descarga (bajando)')
ax.set_xlabel('Masa colgada $m$ [kg]'); ax.set_ylabel('Elongación $\\Delta x$ [cm]')
ax.set_title('Banda elástica: la ida y la vuelta no coinciden')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout(); plt.show()

print("R de la rama de subida :", f"{np.corrcoef(m_banda, subida)[0,1]:.5f}")
print("R de la rama de bajada :", f"{np.corrcoef(m_banda, bajada)[0,1]:.5f}")

Las dos ramas dan $R$ excelentes por separado, y sin embargo el sistema **no tiene** una relación
$\Delta x(m)$: para la misma carga hay dos elongaciones distintas según de dónde venís. Eso se llama
**histéresis**, y significa que la elongación depende de la **historia**, no sólo del estado actual.

Ningún ajuste va a arreglar eso. Ninguna estadística lo va a detectar si sólo medís subiendo. La
única forma de encontrarlo es **medir en las dos direcciones**, y eso es una decisión de diseño que
se toma antes de mirar ningún número.

> Es exactamente el mismo problema que aparece en curvas I–V de junturas memristivas, donde promediar
> la rama de ida con la de vuelta da un resultado muy reproducible y físicamente vacío. La
> reproducibilidad no es evidencia de que el modelo sea correcto.

---
## 8. Aviso para la Clase 10: linealizar deforma los errores

Ya lo viste en el Colab 04 con el ajuste log-log, y conviene dejarlo escrito porque se cobra caro más
adelante.

Si $y' = \ln y$, entonces $\sigma_{y'} = \sigma_y / y$. Un conjunto con **la misma barra absoluta** en
todos los puntos se convierte, al tomar logaritmo, en uno donde los puntos de $y$ chico tienen barras
enormes. Un ajuste no ponderado sobre los datos linealizados les da a todos el mismo peso, y el
resultado queda sesgado.

In [ ]:
tau_verdadero = 2.0
sigma_abs = 0.08                                   # MISMA barra absoluta en todos los puntos
tt = np.linspace(0, 6, 20)

def expo(t, A, tau): return A*np.exp(-t/tau)

# 300 repeticiones del experimento, para ver el SESGO y no una fluctuación
tau_lin_l, tau_nl_l = [], []
for _ in range(300):
    yy = 5*np.exp(-tt/tau_verdadero) + rng.normal(0, sigma_abs, len(tt))
    ok = yy > 0                                    # el log no admite negativos: hay que TIRAR datos
    pl, _ = curve_fit(recta, tt[ok], np.log(yy[ok]))
    tau_lin_l.append(-1/pl[0])
    pn, _ = curve_fit(expo, tt, yy, p0=[5, 2])
    tau_nl_l.append(pn[1])

tau_lin_l, tau_nl_l = np.array(tau_lin_l), np.array(tau_nl_l)
print(f"τ verdadero                     : {tau_verdadero:.4f}")
print(f"τ por log sin pesos   : media = {tau_lin_l.mean():.4f}  "
      f"sesgo = {tau_lin_l.mean()-tau_verdadero:+.4f}  dispersión = {tau_lin_l.std(ddof=1):.4f}")
print(f"τ por ajuste no lineal: media = {tau_nl_l.mean():.4f}  "
      f"sesgo = {tau_nl_l.mean()-tau_verdadero:+.4f}  dispersión = {tau_nl_l.std(ddof=1):.4f}")

Repetimos el experimento 300 veces para separar el **sesgo** de la fluctuación: un solo experimento
no alcanza para distinguirlos, y ése es justamente el error que se comete al comparar dos métodos con
un solo conjunto de datos.

El ajuste sobre los datos linealizados sin pesos está **sesgado**, y además obliga a descartar los
puntos donde el ruido llevó $y$ a valores negativos — datos perfectamente válidos que el logaritmo no
admite, y cuyo descarte no es aleatorio: se van sistemáticamente los de $y$ chico.

Es exactamente lo que pasa al extraer un factor de idealidad de una curva I–V en escala
semilogarítmica: los puntos de corriente baja, que son los peor medidos, terminan dominando el
ajuste.

> **Cuándo linealizar SÍ es correcto.** Si el error es **multiplicativo** (una barra relativa
> constante, como cuando el instrumento tiene un error de escala), entonces el logaritmo es
> exactamente la transformación que vuelve homogéneas las barras, y el ajuste linealizado es el
> apropiado. La regla no es "nunca linealices": es **mirá cómo son tus barras antes de decidir**.

La solución es ajustar en el espacio linealizado **con pesos** (Colab 06) o directamente ajustar el
modelo no lineal (Colab 10).

---
## 9. Ejercicios

**5.4.** Con tus datos: ajustá los dos modelos, reportá $k$ con `reportar()`, mostrá los dos gráficos
con residuos y decidí por escrito cuál modelo adoptás y por qué. La justificación tiene que apoyarse
en los residuos, no en $R^2$.

**5.5.** Determiná tu $F_0$ y verificalo colgando masas chicas (Ejercicio 5.1). Reportá el rango de
validez del modelo sin ordenada: a partir de qué carga la diferencia entre los dos $k$ es menor que
$\sigma_k$.

**5.6.** Agregá a mano un punto claramente erróneo a tus datos (por ejemplo, moviendo una coma) y
volvé a ajustar. ¿Cuánto cambia $k$? ¿Cuánto cambia $R$? ¿Cuál de los dos te avisó del problema?

**5.7.** *(conceptual)* Un compañero informa: "$R^2 = 0{,}9987$, por lo tanto el modelo es correcto".
Escribí en tres oraciones por qué esa afirmación no se sigue, y qué habría que mirar en su lugar.

**5.8.** *(conceptual)* En la Sección 5 viste que el $k$ del modelo correcto es estable frente al
recorte del conjunto y el del modelo incorrecto no. ¿Se te ocurre un caso en que un modelo
**incorrecto** dé igualmente un parámetro estable? (Pista: mirá el conjunto IV de Anscombe.)

**5.9.** Guardá tu resorte identificado. En la Clase 6 vas a medir la misma cosa con repeticiones por
punto, y en la Clase 9 vas a obtener $k$ por un camino completamente distinto. Si cambiás de resorte
en el medio, esas comparaciones no significan nada.